# ReelScribe — Batch Transcription (Kaggle GPU)

**Настройка (один раз):**
1. Add-ons → Secrets → добавь `SUPABASE_URL`, `SUPABASE_ANON_KEY`, `KAGGLE_NOTEBOOK_ID`
2. Settings → Accelerator → **T4 GPU**
3. Settings → Internet → **On**

Запускается автоматически из UI ReelScribe при выборе large-v3.

In [ ]:
!pip install -q faster-whisper yt-dlp imageio-ffmpeg supabase deep-translator

In [ ]:
from kaggle_secrets import UserSecretsClient
_s = UserSecretsClient()
SUPABASE_URL      = _s.get_secret('SUPABASE_URL')
SUPABASE_ANON_KEY = _s.get_secret('SUPABASE_ANON_KEY')

MODEL_NAME   = 'large-v3'
BATCH_LIMIT  = 500
WORKERS      = 4        # параллельных скачиваний
COOKIES_FILE = None     # путь к cookies.txt если Instagram блокирует
TRANSLATE    = True

In [ ]:
from supabase import create_client
db = create_client(SUPABASE_URL, SUPABASE_ANON_KEY)
print('Supabase OK')

In [ ]:
import torch
from faster_whisper import WhisperModel

device       = 'cuda' if torch.cuda.is_available() else 'cpu'
compute_type = 'float16' if device == 'cuda' else 'int8'
print(f'Device: {device} | compute_type: {compute_type}')

model = WhisperModel(MODEL_NAME, device=device, compute_type=compute_type)
print(f'Модель {MODEL_NAME} загружена')

In [ ]:
import os, random, tempfile, time, traceback, threading
from concurrent.futures import ThreadPoolExecutor, as_completed
from datetime import datetime, timedelta, timezone
from pathlib import Path
import yt_dlp, imageio_ffmpeg

AUDIO_DIR = Path('/tmp/reelscribe_audio')
AUDIO_DIR.mkdir(parents=True, exist_ok=True)

# Мьютекс для GPU (транскрипция последовательная, скачивание параллельное)
_gpu_lock = threading.Lock()
_dl_semaphore = threading.Semaphore(WORKERS)


def download_audio(url: str, job_id: str) -> tuple:
    ffmpeg_dir = os.path.dirname(imageio_ffmpeg.get_ffmpeg_exe())
    out_dir = AUDIO_DIR / job_id
    out_dir.mkdir(exist_ok=True)
    opts = {
        'format': 'bestaudio/best',
        'outtmpl': str(out_dir / '%(id)s.%(ext)s'),
        'quiet': True,
        'ffmpeg_location': ffmpeg_dir,
        'postprocessors': [{'key': 'FFmpegExtractAudio', 'preferredcodec': 'wav', 'preferredquality': '0'}],
        'postprocessor_args': {'ffmpeg': ['-ar', '16000', '-ac', '1']},
        'retries': 3,
    }
    if COOKIES_FILE:
        opts['cookiefile'] = COOKIES_FILE
    with yt_dlp.YoutubeDL(opts) as ydl:
        info = ydl.extract_info(url, download=True)
    wavs = sorted(out_dir.glob('*.wav'), key=lambda p: p.stat().st_mtime, reverse=True)
    if not wavs:
        raise FileNotFoundError(f'WAV не найден для {url}')
    return wavs[0], info


def transcribe(wav_path: Path) -> tuple:
    with _gpu_lock:  # GPU используем по одному
        segments, info = model.transcribe(str(wav_path), beam_size=5)
        text = ' '.join(s.text.strip() for s in segments).strip()
        return text, info.language, info.duration


def translate_to_ru(text: str) -> str:
    from deep_translator import GoogleTranslator
    chunks, buf = [], []
    for word in text.split():
        if sum(len(w) for w in buf) + len(word) > 4500:
            chunks.append(' '.join(buf)); buf = []
        buf.append(word)
    if buf: chunks.append(' '.join(buf))
    tr = GoogleTranslator(source='auto', target='ru')
    return ' '.join(tr.translate(c) for c in chunks)


def cleanup_job(job_id: str):
    import shutil
    try: shutil.rmtree(AUDIO_DIR / job_id, ignore_errors=True)
    except: pass


print('Функции готовы')

In [ ]:
now = datetime.now(timezone.utc).replace(tzinfo=None).isoformat()
resp = (
    db.table('jobs')
    .select('*')
    .eq('state', 'queued')
    .lte('next_attempt_at', now)
    .order('next_attempt_at')
    .limit(BATCH_LIMIT)
    .execute()
)
jobs = resp.data
print(f'Джобов в очереди: {len(jobs)}')

In [ ]:
MAX_ATTEMPTS = 3
done_count = 0
fail_count = 0
lock = threading.Lock()


def process_job(job: dict) -> str:
    global done_count, fail_count
    job_id    = job['id']
    reel_id   = job['reel_id']
    session_id= job.get('session_id') or ''
    attempts  = job.get('attempts', 0) + 1

    reel_url = db.table('reels').select('url').eq('id', reel_id).execute().data[0]['url']
    do_translate = TRANSLATE
    if session_id:
        sess = db.table('import_sessions').select('translate').eq('id', session_id).execute().data
        if sess: do_translate = sess[0].get('translate', TRANSLATE)

    db.table('jobs').update({'state': 'in_progress', 'attempts': attempts}).eq('id', job_id).execute()
    db.table('transcripts').update({'status': 'downloading'}).eq('reel_id', reel_id).execute()

    try:
        with _dl_semaphore:  # WORKERS параллельных скачиваний
            wav_path, _ = download_audio(reel_url, job_id)
            time.sleep(random.uniform(1.0, 2.5))

        db.table('transcripts').update({'status': 'transcribing'}).eq('reel_id', reel_id).execute()
        t0 = time.time()
        text, lang, dur = transcribe(wav_path)
        elapsed = time.time() - t0

        text_ru = None
        if do_translate and text and lang and lang != 'ru':
            db.table('transcripts').update({'status': 'translating'}).eq('reel_id', reel_id).execute()
            text_ru = translate_to_ru(text)

        db.table('transcripts').update({
            'text': text, 'text_ru': text_ru, 'language': lang,
            'duration_sec': int(dur), 'status': 'done', 'fail_reason': None,
        }).eq('reel_id', reel_id).execute()
        db.table('jobs').update({'state': 'done', 'error': None}).eq('id', job_id).execute()

        with lock:
            done_count += 1
        return f'✓ {reel_url} ({lang}, {elapsed:.1f}с)'

    except Exception as exc:
        traceback.print_exc()
        next_at = (datetime.utcnow() + timedelta(seconds=30 * attempts)).isoformat()
        state = 'failed' if attempts >= MAX_ATTEMPTS else 'queued'
        db.table('jobs').update({
            'state': state, 'attempts': attempts,
            'next_attempt_at': next_at, 'error': str(exc),
        }).eq('id', job_id).execute()
        db.table('transcripts').update({
            'status': 'failed', 'fail_reason': str(exc)[:500],
        }).eq('reel_id', reel_id).execute()
        with lock:
            fail_count += 1
        return f'✗ {reel_url}: {exc}'

    finally:
        cleanup_job(job_id)


# Запускаем параллельно: WORKERS потоков скачивают, GPU транскрибирует поочерёдно
with ThreadPoolExecutor(max_workers=WORKERS) as executor:
    futures = {executor.submit(process_job, job): job for job in jobs}
    for i, future in enumerate(as_completed(futures), 1):
        result = future.result()
        print(f'[{i}/{len(jobs)}] {result}')

print(f'\n=== Готово: {done_count} успешно, {fail_count} ошибок ===')